In [1]:
### Starting point - Compare two documents to each other (we will need to compare many more documents in production)

In [2]:
# %pip install strands-agents strands-agents-tools fastmcp
# %pip install PyPDF2
# %pip install openpyxl
# %pip install PyMuPDF

In [3]:
import json
import os
from langchain_text_splitters import CharacterTextSplitter
from langchain.docstore.document import Document
from langchain_aws import ChatBedrockConverse
from langchain_aws import BedrockEmbeddings
# from langchain_ollama import ChatOllama
# from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from pathlib import Path
import json, re, uuid
import os
from mcp import StdioServerParameters, stdio_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from PyPDF2 import PdfReader, PdfWriter
import pandas as pd
import sagemaker
import openpyxl
from datetime import date
from OCR_node import OCR, doc_output_path, doc_input_path
import json
from datetime import date
from langchain_core.messages import SystemMessage, HumanMessage
import ast
import fitz
import random

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Fetched defaults config from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


### Check to see if page length of new document is the same as any STR documents

### If so, OCR/BDA the first page of each document

In [4]:
# Assumed input:
veterans_folder = [{"document":"Murphy-STR-Shoulder-PTSD-OSA-Handwritten.pdf","page_no":15,"STR":True, "DBQ":False}, {"document":"Murphy_Scanned.pdf","page_no":15,"STR":True, "DBQ":False}, {"document":"Murphy-STR-scanned.pdf","page_no":15,"STR":True, "DBQ":False}]
uploaded_document = {"document":"Murphy-Radiculopathy-Med-Record-Handwritten.pdf","page_no":15, "STR":True, "DBQ":False} 


######## Uploaded Document = DBQ ########
# veterans_folder = [{"document":"Murphy-STR-Shoulder-PTSD-OSA-Handwritten.pdf","page_no":15,"STR":True, "DBQ":False}, {"document":"Murphy_Scanned.pdf","page_no":15,"STR":True, "DBQ":False}]
# uploaded_document = {"document":"DBQ_test.pdf","page_no":15, "STR":False, "DBQ":True}

####### Uploaded Document = STR #########
# veterans_folder = [{"document":"Murphy-Radiculopathy-Med-Record-Handwritten.json","page_no":15, "STR":True, "DBQ":False}]
# uploaded_document = {"document":"Murphy-STR-Shoulder-PTSD-OSA-Handwritten.pdf","page_no":15, "STR":True, "DBQ":False}

documents_to_check = []
documents_to_check.append(uploaded_document["document"])
for document in veterans_folder:
    if document["page_no"] == uploaded_document["page_no"]:
        documents_to_check.append(document["document"])

In [5]:
# def extract_first_page(pdf_path):
#     reader = PdfReader(pdf_path)
#     first_page_writer = PdfWriter()
#     first_page_writer.add_page(reader.pages[0])
#     with open(Path(f"{doc_input_path}/{Path(pdf_path).stem}_first_page.pdf"), "wb") as output_file:
#         first_page_writer.write(output_file) 
#     return f"{Path(pdf_path).stem}_first_page.pdf"

# document_1_path = "Murphy-Radiculopathy-Med-Record-Handwritten.pdf"
# # document_1_path = "Murphy-Radiculopathy-Med-Record-Handwritten-Copy1.pdf"
# # Example usage
# document_1_page_1 = Path(f"{doc_input_path}/{Path(document_1_path).stem}.pdf")
# document_1_page_1_path = extract_first_page(document_1_page_1)



def extract_first_page(pdf_path):
    reader = PdfReader(pdf_path)
    first_page_writer = PdfWriter()
    first_page_writer.add_page(reader.pages[0])
    page_count = len(reader.pages)
    with open(Path(f"{doc_input_path}/{Path(pdf_path).stem}_first_page.pdf"), "wb") as output_file:
        first_page_writer.write(output_file) 
    return f"{Path(pdf_path).stem}_first_page.pdf"

document_1_path = "Murphy-Radiculopathy-Med-Record-Handwritten.pdf"
# document_1_path = "Murphy-Radiculopathy-Med-Record-Handwritten-Copy1.pdf"
# Example usage
document_1_page_1 = Path(f"{doc_input_path}/{Path(document_1_path).stem}.pdf")
document_1_page_1_path = extract_first_page(document_1_page_1)

# Open the PDF file
upload_doc = fitz.open(Path(f"{doc_input_path}/{Path(uploaded_document["document"])}"))

# Get the number of pages
upload_num_pages = upload_doc.page_count
uploaded_document["page_no"] = upload_num_pages

# Compare page number to the page numbers of other documents in a veteran's folder to find possible duplicates
documents_to_check = []
documents_to_check.append(uploaded_document["document"])
for document in veterans_folder:
    doc_to_check = fitz.open(Path(f"{doc_input_path}/{Path(document["document"])}"))
    folder_num_pages = doc_to_check.page_count
    document["page_no"] = folder_num_pages
    if document["page_no"] == uploaded_document["page_no"]:
        documents_to_check.append(document["document"])

In [6]:
# Get the first page of each document
one_page_documents = []
for document in documents_to_check:
    document_path = Path(f"{doc_input_path}/{Path(document).stem}.pdf")
    one_page_documents.append(extract_first_page(document_path))

In [7]:
from OCR_blueprint import OCR_w_blueprint
# path = "Murphy-Radiculopathy-Med-Record-Handwritten_first_page.pdf"


# document_list = ["first_page_test.pdf", "first_page_test.pdf", "Murphy-STR-Shoulder-PTSD-OSA-Handwritten-Copy1.pdf"] 
document_list = one_page_documents
# document_list = ["DBQ_test.pdf"]

ocr_documents = []
blueprints = []
for document in document_list:
    OCR_w_blueprint(document)
    with open(Path(f"{doc_output_path}/{Path(document).stem}.json"), "r") as f:
        OCRd_file = json.load(f)
    with open(Path(f"{doc_output_path}/{Path(document).stem}_blueprint.json"), "r") as f:
        blueprint = json.load(f)
    ocr_documents.append(OCRd_file)
    blueprints.append(blueprint)

==Running form: Murphy-Radiculopathy-Med-Record-Handwritten_first_page.pdf through OCR==
Document: Murphy-Radiculopathy-Med-Record-Handwritten_first_page_blueprint.json has already been OCR'd
==Running form: Murphy_Scanned_first_page.pdf through OCR==
Document: Murphy_Scanned_first_page_blueprint.json has already been OCR'd


### Determine if new document is a duplicate using Haiku if there is a document with the same page length

In [8]:
from langchain_text_splitters import CharacterTextSplitter
from langchain.docstore.document import Document
from langchain_aws import ChatBedrockConverse
from langchain_aws import BedrockEmbeddings
# from langchain_ollama import ChatOllama
# from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

class LLM_Maker:
    def __init__(self, 
                 chunked_docs,
                 llm,
                 embedding_model,
                 system_prompt):
        
        retriever = FAISS.from_documents(chunked_docs, embedding_model).as_retriever()

        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", (system_prompt)),
                ("human", "{context}\n{input}"),
            ]
        )
        question_answer_chain = create_stuff_documents_chain(llm, prompt)
        self.rag_chain = create_retrieval_chain(retriever, question_answer_chain)


def split_and_chunk_text(input_str):
    all_documents = []
    text_splitter = CharacterTextSplitter(
        # separator="\n\n",
        separator="\n",
        chunk_size=1500,
        chunk_overlap=500,
        length_function=len,
    )
    chunks = text_splitter.split_text(input_str)

    # Convert chunks to Document objects
    for chunk in chunks:
        all_documents.append(Document(page_content=chunk, metadata={}))

    return all_documents


In [9]:
import json
from langchain_core.messages import SystemMessage, HumanMessage

def split_and_chunk_text(text, chunk_size=1000, chunk_overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - chunk_overlap
    return chunks

# ONLY if STR
def compare_documents(document_1_path, document_2_path, chunk_size=1000, chunk_overlap=100):
    large_llm = ChatBedrockConverse(
        model='anthropic.claude-3-5-sonnet-20240620-v1:0',
        temperature=0,
        top_p=0.9
    )

    ruleset_prompt = """
    You are an expert at analyzing and comparing two pages for differences presented.
    You must consider that these documents could have been scanned and have been passed through OCR.
    With that knowledge you must understand that some documents may be the same document,
    but not exact duplicates because of information lost over time or troubles OCRing the data.
    While if a document is very different you shouldn't mark it as the same, you can't ignore if there is enough evidence that a document seems to be similar enough that it can be considered the same.
    Format your output in the following format:
    Name of the new document:
    Date of examination (if applicable):
    'Duplicate?' 'yes' if it is a duplicate, 'no' if it is not.
    """


    documents = ("New document:\n" + document_1_path 
                 + "\n\nOld document:\n" + document_2_path)
    
    chunks = split_and_chunk_text(documents, chunk_size, chunk_overlap)
    input_text = "\n".join(chunks)

    response = large_llm.invoke([
        SystemMessage(content=ruleset_prompt),
        HumanMessage(content=input_text)
    ])


    raw = response.content.strip()
    return raw


#ELSE DBQ checks
# DBQ already exist: yes or no

In [10]:
import re

def extract_duplicate_answer(text):
    """
    Extracts 'yes' or 'no' following 'Duplicate?' (case-insensitive) in the input string.
    Returns the matched answer in lowercase, or None if not found.
    """
    match = re.search(r'Duplicate\?\s*(yes|no)', text, re.IGNORECASE)
    if match:
        return match.group(1).lower()
    return None

In [11]:
document_is_STR = True
if ("DISABILITY BENEFITS QUESTIONNAIRE" in blueprints[0]["explainability_info"][0]['Document Title']['value']): 
    print(True)
    document_is_STR = False

In [12]:
import re

if document_is_STR:
    for i in range(len(ocr_documents) - 1):
        result = compare_documents(ocr_documents[0]['document']['representation']['text'],ocr_documents[i + 1]['document']['representation']['text'])
        print(result)

        duplicate = extract_duplicate_answer(result)
        print()
        # print(duplicate)
        if (duplicate == 'yes'): 
            print()
            print("EARLY RETURN")
            break
        elif (i == (len(ocr_documents) - 2)):
            OCR(uploaded_document["document"])
            with open(Path(f"{doc_output_path}/{Path(uploaded_document["document"]).stem}.json"), "r") as f:
                OCRd_new_file = json.load(f)

            OCRd_new_file['document']['representation']['text']
        title = None
        date = None
else: 
    print("Document is DBQ")
    title = blueprints[0]["explainability_info"][0]['Document Title']['value'] 
    date = blueprints[0]["explainability_info"][0]['Date of Examination']['value']

Name of the new document: Medical Record Consultation Sheet
Date of examination: January 12, 1974
Duplicate? Yes

While there are some minor differences between the two documents, they appear to be the same consultation sheet with slight variations likely due to OCR errors or scanning issues. Here are the key similarities that suggest these are duplicates:

1. The overall structure and format of both documents are identical.
2. The date on both documents is January 12, 1974 (with slight variations in formatting).
3. The location "CampPendletery GA" is identical in both.
4. The reason for request mentions similar symptoms and conditions, including "Brown ciobut/circust" and "Herniafedchsc/oldsc".
5. The consultation report contains very similar text, including phrases like "fremenders pain" and "hurriation/herriation of lower lumber".
6. The prescription "Zxday acitementin" is identical in both.
7. The doctor's name appears to be "Ditauges" or "Drtauges" in both documents.
8. The patien

In [13]:
OCR(uploaded_document["document"])

==Running form: Murphy-Radiculopathy-Med-Record-Handwritten.pdf through OCR==
Document: Murphy-Radiculopathy-Med-Record-Handwritten.json has already been OCR'd


In [14]:
with open(Path(f"{doc_output_path}/{Path(uploaded_document["document"]).stem}.json"), "r") as f:
    OCRd_new_file = json.load(f)

OCRd_new_file['document']['representation']['text']

'MEDICAL RECORD\n\nCONSULTATION SHEET\n\nTO:\nFROM:\nDATE: dm12,1974\n\nREASON FOR REQUEST:\n\nEvatuate fem Rt complains of radiculaphy Brown ciobut\n in hmited flexibility Herniafedchsc?\n\nPROVISIONAL DIAGNOSIS:\n\nSpinalSevasis\n\nDOCTOR:\nDifferes\nLOCATION: CampPendletery GA\n\nCONSULTATION REPORT\n\nPatert in fremenders pain hftry ago seens\n P have caused herriation of lower lumber. Prescrible\n Zxday acitementin\n\nSIGNATURE AND TITLE:\n[SIGNATURE]\nDrtauges\nDATE:\n1/12/1974\n PATIENT IDENTIFICATION:\n[SIGNATURE]\n8m Muphy\n\nCONSULTATION SHEET STANDARD FORM 513 '

### If the new document is a DBQ, compare the type of DBQ to past DBQs to determine if it's a duplicate

In [15]:
if document_is_STR == False:

    ### dummy data
    dbq_history = [{'document_subject': 'DBQ Sleep Apnea', 'document_content_name': 'DBQ-{123-abc}-321-xyz_1.PDF', 'document_name': 'Z321', 'document_type_description': 'Exam', 'document_category_description': 'Records', 'document_type': 356, 'document_source': 'VHA_CAPRI', 'document_path': 'data/'},
                  {'document_subject': 'DBQ Hypertension', 'document_content_name': 'DBQ-{345-abc}-345-xyz_1.PDF', 'document_name': 'Z543', 'document_type_description': 'Exam', 'document_category_description': 'Records', 'document_type': 356, 'document_source': 'VHA_CAPRI', 'document_path': 'data/'},
                   {'document_subject': 'DBQ Back Pain', 'document_content_name': 'DBQ-{678-abc}-678-xyz_1.PDF', 'document_name': 'Z876', 'document_type_description': 'Exam', 'document_category_description': 'Records', 'document_type': 356, 'document_source': 'VHA_CAPRI', 'document_path': 'data/'},
    ]
    
    dbq_subjects = ([x['document_subject'] for x in dbq_history])
    print(dbq_subjects)

    def split_and_chunk_text(text, chunk_size=1000, chunk_overlap=100):
        chunks = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunks.append(text[start:end])
            start += chunk_size - chunk_overlap
        return chunks


    def dbq_match_check(new_dbq_subject, dbq_history, chunk_size=1000, chunk_overlap=100):
        large_llm = ChatBedrockConverse(
            model='anthropic.claude-3-haiku-20240307-v1:0',
            temperature=0,
            top_p=0.01
        )
    
        ruleset_prompt ="""
        You work at the Veterans Affairs office and specialize in determining whether a new Disability Benefits Questionnaire (DBQ) document already exists in a Veteran's DBQ history or not.
        You will be given the subject of the new DBQ document and a list of other DBQ subjects representing the DBQs that already exist in the Veteran's DBQ history.
        Compare the new DBQ's subject with the list of DBQ subjects to determine if it already exists or not.
        Please note, the subjects do not need to match perfectly but they should be very close or should have very similar meaning.
        Output 'DBQ already exists' if the DBQ already exists and 'New DBQ' if it does not exist.
        Only return 'DBQ already exists' or 'New DBQ' in your output.
        """
    
    
        documents = ("New DBQ's subject:\n" + str(new_dbq_subject)
                     + "\n\nVeteran's DBQ history:\n" + str(dbq_history))
    
        chunks = split_and_chunk_text(documents, chunk_size, chunk_overlap)
        input_text = "\n".join(chunks)
    
        response = large_llm.invoke([
            SystemMessage(content=ruleset_prompt),
            HumanMessage(content=input_text)
        ])
    
    
        raw = response.content.strip()
        return raw
    
    dbq_match_determination = dbq_match_check(title, dbq_subjects)
    print(dbq_match_determination)

### Is the uploaded document relevant to one of the Veteran's previous claims?

Given the the metadata of the Veteran's previous claims (primarily the contention, maybe other fields?), which if any relate to the uploaded document

In [16]:
def split_and_chunk_text(text, chunk_size=1000, chunk_overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - chunk_overlap
    return chunks


def fix_OCR(form_to_fix, chunk_size=1000, chunk_overlap=100):
    large_llm = ChatBedrockConverse(
        model='anthropic.claude-3-5-sonnet-20240620-v1:0',
        temperature=0,
        top_p=0.01
    )

    ruleset_prompt = """
    You are an expert at the English language.
    Given a PDF that was scanned using OCR you will translate the document to correct errors.
    The most important thing is keeping the semantic meaning of the document intact.
    You should output nothing but the translated version of the document with your best interpretation after fixing errors.
    """


    document = ("Form to interpret\n" + form_to_fix)
    
    chunks = split_and_chunk_text(document, chunk_size, chunk_overlap)
    input_text = "\n".join(chunks)

    response = large_llm.invoke([
        SystemMessage(content=ruleset_prompt),
        HumanMessage(content=input_text)
    ])


    raw = response.content.strip()
    return raw


In [17]:
# OCRd_new_file is the OCR of the original document:

new_document = OCRd_new_file['document']['representation']['text']
new_document

'MEDICAL RECORD\n\nCONSULTATION SHEET\n\nTO:\nFROM:\nDATE: dm12,1974\n\nREASON FOR REQUEST:\n\nEvatuate fem Rt complains of radiculaphy Brown ciobut\n in hmited flexibility Herniafedchsc?\n\nPROVISIONAL DIAGNOSIS:\n\nSpinalSevasis\n\nDOCTOR:\nDifferes\nLOCATION: CampPendletery GA\n\nCONSULTATION REPORT\n\nPatert in fremenders pain hftry ago seens\n P have caused herriation of lower lumber. Prescrible\n Zxday acitementin\n\nSIGNATURE AND TITLE:\n[SIGNATURE]\nDrtauges\nDATE:\n1/12/1974\n PATIENT IDENTIFICATION:\n[SIGNATURE]\n8m Muphy\n\nCONSULTATION SHEET STANDARD FORM 513 '

In [18]:
file_name = OCRd_new_file['metadata']['s3_key'].split("/")[-1]
file_name

'Murphy-Radiculopathy-Med-Record-Handwritten.pdf'

In [19]:
#############################
### Import Claim Data #######
#############################

# using dummy data for testing purposes
#testing_dictionary = [{"claim_number":"111111", "contention":"PTSD", "closed_date":"07/11/2023"}, {"claim_number":"222222", "contention":"Back Pain", "closed_date":""}, {"claim_number":"33333", "contention":"diabetes mellitus", "closed_date":"02/20/2024"}]
testing_dictionary = [{"claim_number":"111111", "contention":"PTSD", "closed_date":"07/11/2023"}, {"claim_number":"222222", "contention":"Back Pain", "closed_date":""}, {"claim_number":"33333", "contention":"diabetes mellitus", "closed_date":"02/20/2024"}, {"claim_number":"444444", "contention":"hypertension", "closed_date":"07/20/2025"}]

testing_dictionary

[{'claim_number': '111111', 'contention': 'PTSD', 'closed_date': '07/11/2023'},
 {'claim_number': '222222', 'contention': 'Back Pain', 'closed_date': ''},
 {'claim_number': '33333',
  'contention': 'diabetes mellitus',
  'closed_date': '02/20/2024'},
 {'claim_number': '444444',
  'contention': 'hypertension',
  'closed_date': '07/20/2025'}]

In [20]:
def split_and_chunk_text(text, chunk_size=1000, chunk_overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - chunk_overlap
    return chunks


def determine_if_relevant(new_document, claim_data, dbq_examination_date="", chunk_size=1000, chunk_overlap=100):
    large_llm = ChatBedrockConverse(
        model='anthropic.claude-3-5-sonnet-20240620-v1:0',
        temperature=0,
        top_p=0.01
    )

    ruleset_prompt = """
    You work at the Veterans Affairs office and specialize in determining whether an uploaded document relates to an existing claim or not.
    When a new document is uploaded you must compare the text of that document with a dictionary of claims to determine if the document is relevant to any of the claims.
    If there is a match, output the Python dictionary which corresponds to the specific claim and one of the following labels ["No Relationship" or "Guaranteed Relationship"] representing your confidence that the uploaded document is relevant to the claim.
    Output 'No Relationship' if you are confident the document's metadata does not match any of the claims and 'Guaranteed Relationship' if you are 100% confident that they match.
    If the document has a "dbq_examination_date" associated with it, output dbq_date_check = "Pass" if the "dbq_examination_date" is less than or equal to the claim's "closed_date". 
    Output dbq_date_check = "Fail" if the "dbq_examination_date" is after the claim's closed date. Output dbq_date_check = "" (an empty string) if relationship = "No Relationship" or there is no "dbq_examination_date" associated with the document.
    Return your answer in the following structured format:
    If the document's metadata matches at least one of the claims return in this format:
    [{{"claim_data":{{Output the Python dictionary corresponding to this}}, "relationship":[Insert "No Relationship" or "Guaranteed Relationship"], "dbq_examination_date"=[insert dbq_examination_date from input], "dbq_date_check"=["Pass"/"Fail"/""], "summary":[insert 1-2 sentence summary of why or why not the document matches this claim]}}, ... if there is more than claim that is relevant, add those claims in the same dictionary format]
    If the document's metadata does not match any of the claims return a python list containing an empty dictionary e.g. [{{}}]
    """


    documents = ("Uploaded document:\n" + str(new_document)
                 + "\n\ndbq_examination_date:\n" + str(dbq_examination_date)
                 + "\n\nDictionary of claims:\n" + str(claim_data))
    
    chunks = split_and_chunk_text(documents, chunk_size, chunk_overlap)
    input_text = "\n".join(chunks)

    response = large_llm.invoke([
        SystemMessage(content=ruleset_prompt),
        HumanMessage(content=input_text)
    ])


    raw = response.content.strip()
    return raw

relevancy = determine_if_relevant(new_document,testing_dictionary, date)
print(relevancy)

[{"claim_data": {"claim_number": "222222", "contention": "Back Pain", "closed_date": ""}, "relationship": "Guaranteed Relationship", "dbq_examination_date": "<class 'datetime.date'>", "dbq_date_check": "", "summary": "The uploaded document is a medical consultation sheet that discusses a patient's back pain, including mentions of 'radiculaphy', 'herniated disc', and 'lower lumber'. This directly relates to the claim for 'Back Pain'."}]


In [21]:
def extract_claims_list(text):
    # Use regex to extract the list part from the string
    list_str = re.search(r"\[(.|\n)*\]", text).group(0)
    
    # Safely evaluate the string to a Python object
    claims_list = ast.literal_eval(list_str)
    return claims_list

matched_claims_output = extract_claims_list(relevancy)
matched_claims_output

[{'claim_data': {'claim_number': '222222',
   'contention': 'Back Pain',
   'closed_date': ''},
  'relationship': 'Guaranteed Relationship',
  'dbq_examination_date': "<class 'datetime.date'>",
  'dbq_date_check': '',
  'summary': "The uploaded document is a medical consultation sheet that discusses a patient's back pain, including mentions of 'radiculaphy', 'herniated disc', and 'lower lumber'. This directly relates to the claim for 'Back Pain'."}]

In [22]:
### Process the output relevancy/dbq date check node
# If there are any relevent claims, use those in future node
# If there no relevant claims, return the output that got the furthest along in the check

## return an output like below:
## each dictionary in the list corresponds to a separate claim
# relevancy_output = {"determination":"Pass/Fail", "reasoning":"[Insert summary]", "matched_claims":"[insert data of matched claim(s) if applicable]"}

def determine_output(matched_claims_output):
    relevant_claims = []
    irrelevant_claims = []
    relevancy_output = {}

    if matched_claims_output == [{}]:
        relevancy_output["determination"] = "Fail"
        relevancy_output["reasoning"] = f"The new document, '{file_name}', is not relevant to any of the Veteran's claims"
        relevancy_output["matched_claims"] = f""
        return relevancy_output

    else:

        for claim in matched_claims_output:
            # Track the highest response code as this is what will be used for the ouput
            early_return_dict = {1: f"The new document, '{file_name}', is not relevant to any of the Veteran's claims", 2:f"The relevant claim was closed prior to the new DBQ's examination date. The claim was closed on {claim['claim_data']['closed_date']} while the DBQ's examination date is {claim['dbq_examination_date']}. "}
        
            if claim["relationship"]=="No Relationship":
                reasoning_code = 1
                irrelevant_claims.append([claim, reasoning_code])
            elif claim["dbq_date_check"]=="False":
                reasoning_code = 2
                irrelevant_claims.append([claim, reasoning_code, response])
            else:
                relevant_claims.append(claim)

        if relevant_claims != []:
            relevancy_output["determination"] = "Pass"
            relevancy_output["reasoning"] = f"There is at least one relevant claim corresponding to '{file_name}'."
            relevancy_output["matched_claims"] = relevant_claims
            return relevancy_output
        else:
            max_code = 1
            for claim in irrelevant_claims:
                if claim[1] > max_code:
                    max_code = claim[1]
            print("THERE ARE NO RELEVANT CLAIMS")
            
            relevancy_output["determination"] = "Fail"
            relevancy_output["reasoning"] = f"{early_return_dict[max_code]}"
            relevancy_output["matched_claims"] = f""
            return relevancy_output

output = determine_output(matched_claims_output)
output
        

{'determination': 'Pass',
 'reasoning': "There is at least one relevant claim corresponding to 'Murphy-Radiculopathy-Med-Record-Handwritten.pdf'.",
 'matched_claims': [{'claim_data': {'claim_number': '222222',
    'contention': 'Back Pain',
    'closed_date': ''},
   'relationship': 'Guaranteed Relationship',
   'dbq_examination_date': "<class 'datetime.date'>",
   'dbq_date_check': '',
   'summary': "The uploaded document is a medical consultation sheet that discusses a patient's back pain, including mentions of 'radiculaphy', 'herniated disc', and 'lower lumber'. This directly relates to the claim for 'Back Pain'."}]}

### OCR the other STR documents in the Veteran's folder
document_type = 'STR'

In [23]:
# Is it okay to delete this and replace with the cell below?

# veteran_folder = [{"document":"Murphy-favorite-recipes.pdf","STR":False},{"document":"Murphy-STR-Shoulder-PTSD-OSA-Handwritten.pdf","STR":True}, {"document":"Murphy-STR-scanned.pdf","STR":True}]
# OCRd_STRs = []
# for document in veteran_folder:
#     if document["STR"]:
#         OCR(document["document"])
#         with open(Path(f"{doc_output_path}/{Path(uploaded_document["document"]).stem}.json"), "r") as f:
#             OCRd_new_file = json.load(f)
#         OCRd_STRs.append({"name":document["document"],"OCR":OCRd_new_file['document']['representation']['text']})

In [24]:
# veteran_folder = [{"document":"Murphy-favorite-recipes.pdf","STR":False, "DBQ":False},{"document":"Murphy-Radiculopathy-Med-Record-Handwritten.pdf","STR":True, "DBQ":False}, {"document":"Murphy_DBQ CARDIO Hypertension_v10_10_32271b15-af3b-4072-bb3d-0dcb9c2ebdd0_first_page.pdf","STR":False, "DBQ":True}]
OCRd_STRs = []
OCRd_DBQs = []
for document in veterans_folder:
    if document["STR"]:
        OCR(document["document"])
        with open(Path(f"{doc_output_path}/{Path(document["document"]).stem}.json"), "r") as f:
            OCRd_new_file = json.load(f)
        OCRd_STRs.append({"name":document["document"],"OCR":OCRd_new_file['document']['representation']['text']})
    elif document["DBQ"]:
        OCR(document["document"])
        with open(Path(f"{doc_output_path}/{Path(document["document"]).stem}.json"), "r") as f:
            OCRd_new_file = json.load(f)
        OCRd_DBQs.append({"name":document["document"],"OCR":OCRd_new_file['document']['representation']['text']})

==Running form: Murphy-STR-Shoulder-PTSD-OSA-Handwritten.pdf through OCR==
Document: Murphy-STR-Shoulder-PTSD-OSA-Handwritten.json has already been OCR'd
==Running form: Murphy_Scanned.pdf through OCR==
Document: Murphy_Scanned.json has already been OCR'd
==Running form: Murphy-STR-scanned.pdf through OCR==
Document: Murphy-STR-scanned.json has already been OCR'd


In [25]:
OCRd_STRs

[{'name': 'Murphy-STR-Shoulder-PTSD-OSA-Handwritten.pdf',
  'OCR': 'Standard Form 600 General Services Administration FPMR 101-11.806-8\n\nHEALTH RECORD\tCHRONOLOGICAL RECORD OF MEDICAL CARE\nDATE\tSYMPTOMS, DIAGNOSIS, TREATMENT, TREATING ORGANIZATION (Sign each entry)\n6-17-72\tMUTARY 61CK CALL BRANCH chink BARSTON, CA\n\tVITALS: T: 98.5 P: 70 R: 16\n\tB/P: 120/80\n\tSHaLDER PAIN Describulit dall corsta f.\n\tby playny for military lastetbell team while stationed\n\tin RVN. Din Betator cuff likedy. Plans\n\t- mofrin 600mg as neaded daly\n\t- XPAY to dehimine strength of tear\n\t- orthopetic consult\n\t\n9-18-72\tPatiet records indicate recently deployed to HochiMinCity\n\tduring period of intenge lightry. Observed impact of\n\tmusked gas on every combuterts and last members of\n\ttroop to booby trapped bombs. Exhibits symptoms\n\tof PTSD when exposal to lovd noises including vehicle backfires and to smells reminescent\n\tof herbicides used in combet.\n\nFORM 600 Standard Form 600 Gene

In [26]:
OCRd_DBQs

[]

### If the uploaded document is relevant to a claim, is there new information in the document? If so, does it warrant reopening the claim?

In [27]:
# creates a variable for the original STRs OCR. Hard coding for testing purposes.
# also creates a varibale for the ocr of the new STR that caused the EP699.
 
original_str_list = OCRd_STRs

# new_str_output = """
# New:

# Standard Form 600 General Services Administration FPMR 101-11.806-8

# HEALTH RECORD    CHRONOLOGICAL RECORD OF MEDICAL CARE
# DATE    SYMPTOMS, DIAGNOSIS, TREATMENT, TREATING ORGANIZATION (Sign each entry)

# 03-22-75   Follow-up visit
#    Pt reports recurrent low back pain after heavy lifting during training.
#    Describes stiffness in the mornings, occasional radiation to right leg.
#    Pain worsens after long marches, improved slightly with rest.
#    Exam: tenderness over lumbar spine, reduced flexion, no acute neuro deficits.
#    Vitals: BP 130/85, HR 76. Gait guarded but ambulatory.
#    Impression: chronic lumbar strain with possible radiculopathy.

# Plan:
#    - Referred for lumbar spine x-rays
#    - Prescribed muscle relaxant and rest profile for 14 days
#    - PT referral for strengthening/stretching program
#    - If symptoms persist, order MRI to rule out disc involvement
#    - Noted progressive functional impact compared to prior STRs

# FORM 600 Standard Form 600 General Services Administration FPMR 101-11.806-8
# """
# new_str = {"name": "Dummy_STR", "OCR": new_str_output}
if document_is_STR:
    new_str_output = new_document
    new_str = {"name": "Dummy_STR", "OCR": new_str_output}
else: 
    dummy_dbq_output = new_document
    new_str = {"name": "Dummy_DBQ", "OCR": dummy_dbq_output}
# dummy_dbq_output = """
# Department of Veterans Affairs

# HYPERTENSION DISABILITY BENEFITS QUESTIONNAIRE

# Name of Claimant/Veteran: SAMUEL MURPHY
# Date of Examination: 04/12/2025

# SECTION I - DIAGNOSIS
# 1A. Current diagnosis: Essential Hypertension
#    ICD code: I10
#    Date of diagnosis: 2010 (per service records review)

# SECTION II - MEDICAL HISTORY
# 2A. History: Veteran reports long-standing elevated blood pressure since late 1970s,
#    initially treated sporadically with hydrochlorothiazide. Condition has gradually
#    worsened; currently requires combination therapy.
#    Veteran denies hospitalization for hypertension but reports intermittent dizziness,
#    headaches, and blurred vision.
#    Current medications: lisinopril 20mg daily, amlodipine 10mg daily.

# 2B. Treatment plan requires continuous medication: YES

# 2C. Initial diagnosis confirmed by multiple elevated BP readings during service.

# SECTION III - CURRENT BLOOD PRESSURE READINGS
# Date: 04/12/2025
#    Reading #1: 162/96
#    Reading #2: 158/94
#    Reading #3: 166/98

# SECTION IV - COMPLICATIONS / FUNCTIONAL IMPACT
#    - Veteran reports difficulty with strenuous activity due to fatigue and headaches.
#    - At risk for cardiovascular complications, especially with history of elevated
#      cholesterol and family history of stroke.
#    - Functional impact: limited ability to perform prolonged physical exertion.

# SECTION V - REMARKS
#    Veteran’s hypertension remains uncontrolled despite dual therapy.
#    Recommend ongoing management, dietary modification, and further cardiology consult.

# Examiner: Dr. Jane Doe, MD
# NPI: 98765432
# Date signed: 04/12/2025
# """

# dummy_dbq = {"name": "Dummy_DBQ", "OCR": dummy_dbq_output}

In [28]:
def summarize_all_documents(str_list):
    """
    Summarize STRs and DBQs into a single JSON structure.
    Works with any number of STRs and DBQs.
    Returns:
    {
      "STRs": { "STR 1": {...}, "STR 2": {...}, ... },
      "DBQs": { "DBQ 1": {...}, "DBQ 2": {...}, ... }
    }
    """
    large_llm = ChatBedrockConverse(
        model="anthropic.claude-3-5-sonnet-20240620-v1:0",
        temperature=0,
        top_p=0.01
    )

    str_ruleset = """
    You are a VA claims adjudicator.
    Summarize the OCR'd Service Treatment Record (STR).

    Rules:
    - Keep ONLY claims-relevant info (diagnoses, symptoms, treatments, functional impact).
    - Ignore boilerplate (headers, form labels, vitals unless medically relevant).
    - Output STRICT JSON:
      {
        "diagnoses": [...],
        "symptoms": [...],
        "treatments": [...],
        "functional_impact": "...",
        "timeline_notes": "..."
      }
    """

    dbq_ruleset = """
    You are a VA claims adjudicator.
    Summarize the OCR'd Disability Benefits Questionnaire (DBQ).

    Rules:
    - Keep ONLY claims-relevant info:
        • diagnoses (ICD codes if listed)
        • symptoms/history
        • treatments/medications
        • complications
        • functional impact
        • examiner remarks
    - Ignore boilerplate (headers, examiner address, certification language).
    - Output STRICT JSON:
      {
        "diagnoses": [...],
        "symptoms_history": "...",
        "treatments": [...],
        "complications": [...],
        "functional_impact": "...",
        "examiner_remarks": "..."
      }
    """

    results = {"STRs": {}}

    # summarize STRs
    for idx, doc in enumerate(str_list, start=1):
        response = large_llm.invoke([
            SystemMessage(content=str_ruleset),
            HumanMessage(content=f"STR Document {idx}:\n\n{doc['OCR']}")
        ])
        raw = response.content.strip()
        try:
            start = raw.index("{")
            end = raw.rindex("}") + 1
            parsed = json.loads(raw[start:end])
            results["STRs"][f"STR {idx}"] = parsed
        except Exception:
            results["STRs"][f"STR {idx}"] = {"error": "could not parse JSON", "raw_output": raw}

    # summarize DBQs
    # for idx, doc in enumerate(dbq_list, start=1):
    #     response = large_llm.invoke([
    #         SystemMessage(content=dbq_ruleset),
    #         HumanMessage(content=f"DBQ Document {idx}:\n\n{doc['OCR']}")
    #     ])
    #     raw = response.content.strip()
    #     try:
    #         start = raw.index("{")
    #         end = raw.rindex("}") + 1
    #         parsed = json.loads(raw[start:end])
    #         results["DBQs"][f"DBQ {idx}"] = parsed
    #     except Exception:
    #         results["DBQs"][f"DBQ {idx}"] = {"error": "could not parse JSON", "raw_output": raw}

    return results


# --- example usage ---
if __name__ == "__main__":
    str_docs = [
        {"OCR": "…OCR text for first STR"}
        # {"OCR": "…OCR text for second STR…"}
    ]
    #dbq_docs = [
    #    {"OCR": "…OCR text for first DBQ…"}
    #]
    summaries = summarize_all_documents(str_docs)
    print(json.dumps(summaries, indent=2))


{
  "STRs": {
    "STR 1": {
      "diagnoses": [
        "Chronic low back pain",
        "Lumbar strain"
      ],
      "symptoms": [
        "Low back pain",
        "Pain radiating to left leg",
        "Numbness in left foot"
      ],
      "treatments": [
        "Physical therapy",
        "NSAIDs",
        "Muscle relaxants"
      ],
      "functional_impact": "Difficulty with prolonged standing, walking, and lifting heavy objects. Limited range of motion in lower back.",
      "timeline_notes": "Patient first reported low back pain in June 2018 after heavy lifting incident. Symptoms worsened over time, with radiating pain and numbness developing by January 2019."
    }
  }
}


In [1]:
def evaluate_new_vs_old_strs(old_strs, new_str, claim):
    old_text = "\n\n".join([f"Document: {d['name']}\n{d['OCR']}" for d in old_strs])
    new_text = f"Document: {new_str['name']}\n{new_str['OCR']}"

    large_llm = ChatBedrockConverse(
        model="anthropic.claude-3-5-sonnet-20240620-v1:0",
        temperature=0,
        top_p=0.1
    )

    ruleset_prompt = f"""
    You are a VA claims adjudicator.

    Claim under review:
    - Claim Number: {claim['claim_data']['claim_number']}
    - Contention: {claim['claim_data']['contention']}
    - Relationship: {claim['relationship']}
    - Prior Summary: {claim['summary']}

    Task:
    - Review the PRIOR STRs.
    - Review the NEW STR.
    - Determine MATERIALITY: does the new STR add evidence that could reasonably change the outcome of this claim?

    Output in strict JSON only with these fields:
    {{
      "Claim Number": "{claim['claim_data']['claim_number']}",
      "Contention": "{claim['claim_data']['contention']}",
      "Recommendation": "Increase/Close",
      "Change": "Succinct categorization of what changed",
      "Claim Note Update": "Specific VBMS note update if required",
      "Overall Rationale": "Short explicit explanation of what new evidence the new STR provides.",
      "Original STR Synopsis": "Synthesized history of the original STRs"
    }}
    """

    response = large_llm.invoke([
        SystemMessage(content=ruleset_prompt),
        HumanMessage(content=f"PRIOR STRs:\n{old_text}\n\nNEW STR:\n{new_text}")
    ])

    raw = response.content.strip()
    try:
        start = raw.index("{")
        end = raw.rindex("}") + 1
        return json.loads(raw[start:end])
    except Exception:
        return {"error": "could not parse JSON", "raw_output": raw}


def evaluate_new_vs_old_dbqs(old_dbqs, new_dbq, claim):
    old_text = "\n\n".join([f"Document: {d['name']}\n{d['OCR']}" for d in old_dbqs])
    new_text = f"Document: {new_dbq['name']}\n{new_dbq['OCR']}"

    large_llm = ChatBedrockConverse(
        model="anthropic.claude-3-5-sonnet-20240620-v1:0",
        temperature=0,
        top_p=0.1
    )

    ruleset_prompt = f"""
    You are a VA claims adjudicator.

    Claim under review:
    - Claim Number: {claim['claim_data']['claim_number']}
    - Contention: {claim['claim_data']['contention']}
    - Relationship: {claim['relationship']}
    - Prior Summary: {claim['summary']}

    Task:
    - Review the PRIOR DBQs.
    - Review the NEW DBQ.
    - Determine MATERIALITY: does the new DBQ add evidence that could reasonably change the outcome of this claim?

    Output in strict JSON only with these fields:
    {{
      "Claim Number": "{claim['claim_data']['claim_number']}",
      "Contention": "{claim['claim_data']['contention']}",
      "Recommendation": "Increase/Close",
      "Change": "Succinct categorization of what changed",
      "Claim Note Update": "Specific VBMS note update if required",
      "Overall Rationale": "Short explicit explanation of what new evidence the new DBQ provides.",
      "Original DBQ Synopsis": "Synthesized history of the original DBQs"
    }}
    """

    response = large_llm.invoke([
        SystemMessage(content=ruleset_prompt),
        HumanMessage(content=f"PRIOR DBQs:\n{old_text}\n\nNEW DBQ:\n{new_text}")
    ])

    raw = response.content.strip()
    try:
        start = raw.index("{")
        end = raw.rindex("}") + 1
        return json.loads(raw[start:end])
        
    except Exception:
        return {"error": "could not parse JSON", "raw_output": raw}

# --- usage ---
if __name__ == "__main__":
    old_strs = OCRd_STRs
    # old_dbqs = OCRd_DBQs

    evaluations = {
        "STRs": [evaluate_new_vs_old_strs(old_strs, new_str, claim) for claim in matched_claims_output]
        # "DBQs": [evaluate_new_vs_old_dbqs(old_dbqs, dummy_dbq, claim) for claim in matched_claims_output]
    }

    print(json.dumps(evaluations, indent=2))


NameError: name 'OCRd_STRs' is not defined

In [30]:
evaluations = evaluations['STRs']

In [35]:
# for i in range(len(evaluations)):
#     if evaluations[i]["Recommendation"] == "Increase":
#         action_taken = "The new document contains information which likely warrants an increase in rating percentage. Recommondation is to reopen the claim."
#     else:
#         action_taken = "The new dodcument does not contain any new information. Recommondation is to close the claim."
    
#     evaluations[i]['claim_note'] = {'extItemId': '135792468',
#        'extItemType': 'CLAIM',
#        'narrative': evaluations[i]["Overall Rationale"],
#        'actionTaken': action_taken,
#        'jpaVersion': 1,
#        'editable': False,
#        'noteType': 'VBMS_PERMANENT',
#        'systemGenerated': True,
#        'realm': 'VBA'}
    
# print(evaluations)

for i in range(len(evaluations)):
    if not evaluations[i]:
        continue

    doc_name = evaluations[i].get("Document Name", "<unknown>")
    if not doc_name.lower().endswith(".pdf"):
        doc_name += ".pdf"
    evaluations[i]["Document Name"] = doc_name

    rec = evaluations[i].get("Recommendation", "")
    contention = evaluations[i].get("Contention", "<unknown>")

    base_claim_note = {
        "extItemId": "135792468",
        "extItemType": "CLAIM",
        "jpaVersion": 1,
        "editable": False,
        "noteType": "VBMS_PERMANENT",
        "systemGenerated": True,
        "realm": "VBA"
    }

    if rec == "Increase":
        claim_number = evaluations[i-1]['Claim Number']
        action_taken = (
            "The new document contains information which likely warrants an "
            "increase in rating percentage. Recommendation is to reopen the claim."
        )
        narrative = (
            f"{doc_name} contains evidence of new information related to '{contention}' that would have been available when rating Claim Number {claim_number}. "
            f"Total page count of new document: {folder_num_pages}. SMART AGENT"
        )
        claim_note_dict = {
            **base_claim_note,
            "narrative": narrative,
            "actionTaken": action_taken
        }

        evaluations[i]["claim_note"] = claim_note_dict
        evaluations[i]["Notes"] = (
            f"SMART AGENT review identified new information.  See claim notes for details."
        )

    else:
        action_taken = (
            "The new document does not contain any new information. "
            "Recommendation is to close the claim."
        )
        claim_note_dict = {
            **base_claim_note,
            "actionTaken": action_taken
        }

        evaluations[i]["claim_note"] = claim_note_dict
        evaluations[i]["Notes"] = (
            f"SMART AGENT review did not identify new information relevant to the Veteran's claim(s)."
        ) 

print(evaluations)

# print doc id, narrative, and note

[{'Claim Number': '222222', 'Contention': 'Back Pain', 'Recommendation': 'Increase', 'Change': 'New evidence of radiculopathy and herniated disc', 'Claim Note Update': 'New STR dated 1/12/1974 shows diagnosis of radiculopathy and herniated disc in lower lumbar region, supporting claim for back pain.', 'Overall Rationale': 'The new STR provides specific diagnoses (radiculopathy and herniated disc) that were not present in the original STRs, offering stronger evidence for the back pain claim.', 'Original STR Synopsis': 'Original STRs mentioned shoulder pain, PTSD symptoms, chest discomfort, and sleep apnea, but lacked specific evidence related to back pain.', 'Document Name': '<unknown>.pdf', 'claim_note': {'extItemId': '135792468', 'extItemType': 'CLAIM', 'jpaVersion': 1, 'editable': False, 'noteType': 'VBMS_PERMANENT', 'systemGenerated': True, 'realm': 'VBA', 'narrative': "<unknown>.pdf contains evidence of new information related to 'Back Pain' that would have been available when rati

In [36]:
summary_output = []

for ev in evaluations:
    if not ev:
        continue

    doc_id = str(random.randint(100000, 999999))
    narrative_text = ev.get("claim_note", {}).get("narrative", "")
    note_text = ev.get("Notes", "")

    entry = {
        "Document ID": doc_id,
        "Notes": note_text
    }

    if narrative_text:
        entry["Narrative"] = narrative_text

    summary_output.append(entry)

print(json.dumps(summary_output, indent=2))


[
  {
    "Document ID": "664687",
    "Notes": "SMART AGENT review identified new information.  See claim notes for details.",
    "Narrative": "<unknown>.pdf contains evidence of new information related to 'Back Pain' that would have been available when rating Claim Number 222222. Total page count of new document: 3. SMART AGENT"
  }
]


In [32]:
# def driver(veterans_folder, uploaded_document):
#     documents_to_check = []
#     documents_to_check.append(uploaded_document["document"])
#     for document in veterans_folder:
#         if document["page_no"] == uploaded_document["page_no"]:
#             documents_to_check.append(document["document"])
#     if len(documents_to_check) > 1:

#         # Get the first page of each document
#         one_page_documents = []
#         for document in documents_to_check:
#             document_path = Path(f"{doc_input_path}/{Path(document).stem}.pdf")
#             one_page_documents.append(extract_first_page(document_path))

#         document_list = one_page_documents

#         ocr_documents = []
#         for document in document_list:
#             OCR(document)
#             with open(Path(f"{doc_output_path}/{Path(document).stem}.json"), "r") as f:
#                 OCRd_file = json.load(f)
#             ocr_documents.append(OCRd_file)
        
#         import re
#         for i in range(len(ocr_documents) - 1):
#             result = compare_documents(ocr_documents[0]['document']['representation']['text'],ocr_documents[i + 1]['document']['representation']['text'], "Comparing the two documents pages analyze if they are duplicates").get("answer")
#             print(result)

#             duplicate = extract_duplicate_answer(result)
#             print()
#             # print(duplicate)
#             if (duplicate == 'yes'): 
#                 print()
#                 print("Document is a duplicate of a previous record in their file.")
#                 return
    
#     # We get here if:
#         # One: None of the uploaded documents have the same page number as the uploaded one.
#         # Two: We have checked the first page of each document that has the same page no. and it's not a duplicate
                    
#     print("No documents have the same page number")
#     OCR(uploaded_document["document"])
#     with open(Path(f"{doc_output_path}/{Path(uploaded_document["document"]).stem}.json"), "r") as f:
#         OCRd_new_file = json.load(f)

#     OCRd_new_file['document']['representation']['text']

        
# veterans_folder = [{"document":"Murphy-favorite-recipes.pdf","page_no":100, "STR": False}, {"document":"Murphy_Scanned.pdf","page_no":15,"STR":False}, {"document":"Murphy-STR-scanned.pdf","page_no":15, "STR":True}]
# uploaded_document = {"document":"Murphy-STR-Shoulder-PTSD-OSA-Handwritten.pdf","page_no":15} 